In [1]:
# Step 1: Import required libraries and modules
import sys, os
sys.path.append(os.path.abspath("..")) 

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import json
from datetime import datetime

from models.mobilenetv3 import MobileNetV3Extractor
from models.lstm_attention import BiLSTMWithAttention
from preprocessing.dataset import SignLanguageDataset

In [2]:
# Step 2: Setup device and data paths

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Data paths
train_root = "../data/frames/train"
val_root = "../data/frames/validation"

# Load label mapping
class_names = sorted([
    d for d in os.listdir(train_root)
    if os.path.isdir(os.path.join(train_root, d)) and not d.startswith(".")
])
label_map = {name: idx for idx, name in enumerate(class_names)}
inv_label_map = {v: k for k, v in label_map.items()}

print(f"Total classes: {len(label_map)}")
print("First 10 classes:", list(label_map.items())[:10])

In [3]:
# Step 3: Define model structures and loading functions

class FullSLRModel(nn.Module):
    def __init__(self, num_classes, hidden_dim=128, num_layers=2):
        super().__init__()
        self.feature_extractor = MobileNetV3Extractor()
        self.temporal_model = BiLSTMWithAttention(
            input_dim=960, 
            hidden_dim=hidden_dim, 
            num_classes=num_classes,
            num_layers=num_layers
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        features = self.feature_extractor(x)
        features = features.view(B, T, -1)
        logits, attention_weights = self.temporal_model(features)
        return logits, attention_weights

class MobileNetV3Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.feature_extractor = MobileNetV3Extractor()
        self.fc = nn.Linear(960, num_classes)

    def forward(self, x):
        if x.dim() == 5:
            x = x[:, 0, :, :, :]  # Take first frame only
        features = self.feature_extractor(x)
        logits = self.fc(features)
        return logits, None  # Return None for attention weights for compatibility

def load_model(model_class, model_path, num_classes, device):
    """Load a trained model from checkpoint"""
    model = model_class(num_classes=num_classes).to(device)
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Model loaded from {model_path}")
    else:
        print(f"Warning: Model file {model_path} not found!")
    return model

In [ ]:
# Step 4: validation set and test set data.

val_transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# validation set
val_dataset = SignLanguageDataset(
    root_dir=val_root,
    label_map=label_map,
    transform=val_transform
)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

# test set
test_root = "../data/frames/test"
test_dataset = SignLanguageDataset(
    root_dir=test_root,
    label_map=label_map,
    transform=val_transform
)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

In [6]:
# Step 5: Define performance evaluation functions

def evaluate_model_performance(model, val_loader, device, model_name="Model"):
    """Comprehensive model performance evaluation"""
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    all_attention_weights = []
    
    with torch.no_grad():
        for videos, labels in tqdm(val_loader, desc=f'Evaluating {model_name}'):
            videos = videos.to(device)
            labels = labels.to(device)
            
            outputs, attention_weights = model(videos)
            probabilities = torch.softmax(outputs, dim=1)
            
            _, predicted = outputs.max(1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            
            if attention_weights is not None:
                all_attention_weights.extend(attention_weights.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    
    # Per-class metrics
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        all_labels, all_predictions, average=None, zero_division=0
    )
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    results = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'support_per_class': support_per_class,
        'confusion_matrix': cm,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probabilities,
        'attention_weights': all_attention_weights if all_attention_weights else None
    }
    
    return results

def print_performance_summary(results, model_name):
    """Print detailed performance summary"""
    print(f"\n{'='*60}")
    print(f"PERFORMANCE SUMMARY: {model_name}")
    print(f"{'='*60}")
    print(f"Overall Accuracy: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")
    print(f"Macro Precision: {results['precision']:.4f}")
    print(f"Macro Recall: {results['recall']:.4f}")
    print(f"Macro F1-Score: {results['f1_score']:.4f}")
    print(f"\nPer-class metrics (showing first 10 classes):")
    print(f"{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1':<10} {'Support':<10}")
    print("-" * 60)
    for i in range(min(10, len(results['precision_per_class'])):
        class_name = inv_label_map[i]
        print(f"{class_name:<15} {results['precision_per_class'][i]:<10.4f} "
              f"{results['recall_per_class'][i]:<10.4f} {results['f1_per_class'][i]:<10.4f} "
              f"{results['support_per_class'][i]:<10.0f}")

RuntimeError: Error(s) in loading state_dict for FullSLRModel:
	size mismatch for temporal_model.classifier.weight: copying a param with shape torch.Size([301, 512]) from checkpoint, the shape in current model is torch.Size([293, 512]).
	size mismatch for temporal_model.classifier.bias: copying a param with shape torch.Size([301]) from checkpoint, the shape in current model is torch.Size([293]).

In [ ]:
# Step 6: Load and evaluate all models

# Define model paths (adjust these based on your actual saved models)
model_paths = {
    'MobileNetV3_Only': 'mobilenetv3_baseline_best.pth',
    'Experiment1_Basic': 'experiment1_best_model.pth',
    'Experiment2_Augmented': 'experiment2_best_model.pth'
}

# Define model classes
model_classes = {
    'MobileNetV3_Only': MobileNetV3Classifier,
    'Experiment1_Basic': FullSLRModel,
    'Experiment2_Augmented': FullSLRModel
}

# Evaluate all models
all_results = {}

for model_name, model_path in model_paths.items():
    print(f"\nEvaluating {model_name}...")
    
    # Load model
    model_class = model_classes[model_name]
    model = load_model(model_class, model_path, len(label_map), device)
    
    # Evaluate performance
    results = evaluate_model_performance(model, val_loader, device, model_name)
    all_results[model_name] = results
    
    # Print summary
    print_performance_summary(results, model_name)

In [ ]:
# Step 7: Confusion matrix visualization

def plot_confusion_matrix(cm, class_names, model_name, max_classes=20):
    """Plot confusion matrix for a model"""
    if len(class_names) > max_classes:
        # For large number of classes, show top classes by support
        class_support = cm.sum(axis=1)
        top_indices = np.argsort(class_support)[-max_classes:]
        cm = cm[top_indices][:, top_indices]
        class_names = [class_names[i] for i in top_indices]
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Plot confusion matrices for all models
for model_name, results in all_results.items():
    print(f"\nPlotting confusion matrix for {model_name}...")
    class_names_list = [inv_label_map[i] for i in range(len(label_map))]
    plot_confusion_matrix(results['confusion_matrix'], class_names_list, model_name)

In [ ]:
# Step 8: Performance metrics comparison visualization

def plot_performance_comparison(all_results):
    """Plot performance comparison across all models"""
    models = list(all_results.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1_score']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.ravel()
    
    for i, metric in enumerate(metrics):
        values = [all_results[model][metric] for model in models]
        
        bars = axes[i].bar(models, values, color=['lightblue', 'lightcoral', 'lightgreen'])
        axes[i].set_title(f'{metric.replace("_", " ").title()}')
        axes[i].set_ylabel(metric.replace("_", " ").title())
        axes[i].set_ylim(0, max(values) * 1.1)
        
        # Add value labels on bars
        for bar, value in zip(bars, values):
            axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                        f'{value:.3f}', ha='center', va='bottom')
        
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

# Plot performance comparison
plot_performance_comparison(all_results)

In [ ]:
# Step 9: Error analysis

def analyze_error_cases(all_results, val_dataset, num_cases=10):
    """Analyze cases where models made errors"""
    print(f"\n{'='*60}")
    print("ERROR ANALYSIS")
    print(f"{'='*60}")
    
    for model_name, results in all_results.items():
        print(f"\n{model_name} Error Analysis:")
        
        # Find indices where predictions don't match labels
        predictions = np.array(results['predictions'])
        labels = np.array(results['labels'])
        error_indices = np.where(predictions != labels)[0]
        
        print(f"Total errors: {len(error_indices)} out of {len(labels)} ({len(error_indices)/len(labels)*100:.2f}%)")
        
        # Show first few error cases
        for i in range(min(num_cases, len(error_indices))):
            idx = error_indices[i]
            true_label = labels[idx]
            pred_label = predictions[idx]
            true_class = inv_label_map[true_label]
            pred_class = inv_label_map[pred_label]
            
            print(f"  Case {i+1}: True={true_class}({true_label}) -> Pred={pred_class}({pred_label})")
        
        # Most common error patterns
        error_pairs = [(labels[i], predictions[i]) for i in error_indices]
        from collections import Counter
        most_common_errors = Counter(error_pairs).most_common(5)
        print(f"\nMost common error patterns:")
        for (true_l, pred_l), count in most_common_errors:
            true_class = inv_label_map[true_l]
            pred_class = inv_label_map[pred_l]
            print(f"  {true_class} -> {pred_class}: {count} times")

# Run error analysis
analyze_error_cases(all_results, val_dataset)

In [ ]:
# Step 10: Attention weights visualization

def visualize_attention_weights(model, val_loader, device, num_samples=5):
    """Visualize attention weights for models with attention mechanism"""
    model.eval()
    
    with torch.no_grad():
        for batch_idx, (videos, labels) in enumerate(val_loader):
            if batch_idx >= num_samples:
                break
                
            videos = videos.to(device)
            outputs, attention_weights = model(videos)
            
            if attention_weights is not None:
                for i in range(min(2, videos.size(0))):  # Show first 2 samples
                    video = videos[i]  # [T, C, H, W]
                    attention = attention_weights[i]  # [T, 1]
                    
                    # Plot attention weights over time
                    plt.figure(figsize=(12, 4))
                    
                    plt.subplot(1, 2, 1)
                    frames = range(len(attention))
                    plt.bar(frames, attention.squeeze())
                    plt.title(f'Attention Weights - Sample {batch_idx*4 + i}')
                    plt.xlabel('Frame')
                    plt.ylabel('Attention Weight')
                    
                    # Show some frames with attention
                    plt.subplot(1, 2, 2)
                    # Convert video tensor to image for display
                    frame_idx = attention.argmax().item()
                    frame = video[frame_idx].cpu().permute(1, 2, 0)
                    # Denormalize
                    mean = torch.tensor([0.485, 0.456, 0.406])
                    std = torch.tensor([0.229, 0.224, 0.225])
                    frame = frame * std + mean
                    frame = torch.clamp(frame, 0, 1)
                    plt.imshow(frame)
                    plt.title(f'Frame {frame_idx} (Max Attention)')
                    plt.axis('off')
                    
                    plt.tight_layout()
                    plt.show()

# Visualize attention for models with attention mechanism
for model_name in ['Experiment1_Basic', 'Experiment2_Augmented']:
    if model_name in all_results and all_results[model_name]['attention_weights']:
        print(f"\nVisualizing attention weights for {model_name}...")
        model = load_model(FullSLRModel, model_paths[model_name], len(label_map), device)
        visualize_attention_weights(model, val_loader, device)

In [ ]:
# Step 11: Save evaluation results

def save_evaluation_results(all_results):
    """Save all evaluation results to JSON file"""
    # Convert numpy arrays to lists for JSON serialization
    results_for_save = {}
    for model_name, results in all_results.items():
        results_for_save[model_name] = {
            'accuracy': float(results['accuracy']),
            'precision': float(results['precision']),
            'recall': float(results['recall']),
            'f1_score': float(results['f1_score']),
            'precision_per_class': results['precision_per_class'].tolist(),
            'recall_per_class': results['recall_per_class'].tolist(),
            'f1_per_class': results['f1_per_class'].tolist(),
            'support_per_class': results['support_per_class'].tolist(),
            'confusion_matrix': results['confusion_matrix'].tolist(),
            'predictions': results['predictions'],
            'labels': results['labels']
        }
    
    # Add metadata
    results_for_save['metadata'] = {
        'evaluation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'num_classes': len(label_map),
        'class_names': list(label_map.keys()),
        'device': str(device)
    }
    
    # Save to file
    with open('performance_evaluation_results.json', 'w') as f:
        json.dump(results_for_save, f, indent=2)
    
    print("\nEvaluation results saved to 'performance_evaluation_results.json'")

# Save results
save_evaluation_results(all_results)

# Print final summary
print(f"\n{'='*60}")
print("FINAL PERFORMANCE SUMMARY")
print(f"{'='*60}")
print(f"{'Model':<25} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1-Score':<10}")
print("-" * 70)
for model_name, results in all_results.items():
    print(f"{model_name:<25} {results['accuracy']:<10.4f} {results['precision']:<10.4f} "
          f"{results['recall']:<10.4f} {results['f1_score']:<10.4f}")